In [ ]:
# 환경 세팅

# 시스템 패키지 설치
system("apt-get update -qq")
system("apt-get install -y libxml2-dev libcurl4-openssl-dev libssl-dev build-essential")

# R 재설치 환경 정리
unlink("/usr/local/lib/R/site-library/00LOCK*", recursive = TRUE)

# BiocManager 설치
install.packages("BiocManager", repos="https://cloud.r-project.org")

# Dependency 설치
BiocManager::install("S4Vectors", ask=FALSE, update=FALSE)
BiocManager::install("IRanges", ask=FALSE, update=FALSE)
BiocManager::install("GenomeInfoDb", ask=FALSE, update=FALSE)
BiocManager::install("SummarizedExperiment", ask=FALSE, update=FALSE)

# DESeq2 설치
BiocManager::install("DESeq2", ask=FALSE, update=FALSE)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Installing package(s) 'BiocVersion', 'S4Vectors'

also installing the dependency ‘BiocGenerics’


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Installing package(s) 'IRanges'

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04

In [ ]:
# 필요한 라이브러리 로드
BiocManager::install("limma", ask=FALSE, update=FALSE) # limma 패키지 설치
library(DESeq2)
library(limma)

# 1. 통합 CSV 파일 불러오기
# 행(Row)이 유전자(gene_id)이므로 row.names = 1로 지정합니다.
counts_matrix <- read.csv("tcga_cancer_normal_counts.csv", row.names = 1, check.names = FALSE)

# 2. 샘플 ID(열 이름)로부터 자동으로 Target과 Batch 추출하기
sample_ids <- colnames(counts_matrix)

# (1) 암/정상 구별 (Sample Type 코드 추출)
# TCGA ID에서 4번째 세트(예: 01A, 11A)의 숫자 두 자리를 가져옵니다.
sample_types <- sapply(strsplit(sample_ids, "-"), `[`, 4)
sample_type_num <- substr(sample_types, 1, 2)

# 01~09는 Cancer, 11은 Normal로 분류
target <- ifelse(as.numeric(sample_type_num) < 10, "cancer", "normal")

# (2) 배치(Batch) 구별 (Plate ID 추출)
# TCGA ID에서 6번째 세트(예: 1755, A40A 등 바로 앞의 플레이트 바코드 세트)를 배치로 사용합니다.
# 안전하게 5번째 세트(Plate/Center 영역 전체)를 가져오기 위해 5번째 마디를 추출합니다.
batches <- sapply(strsplit(sample_ids, "-"), `[`, 5)

# 3. 메타데이터 데이터프레임 생성
metadata <- data.frame(
  row.names = sample_ids,
  target = as.factor(target),
  batch = as.factor(batches)
)

# DESeq2는 정수형 matrix를 원하므로 형변환
counts_matrix <- as.matrix(counts_matrix)
class(counts_matrix) <- "integer"

print("--- 샘플 정보 자동 추출 완료 ---")
print(table(metadata$target)) # 암/정상 샘플 수 확인

# 4. DESeq2 객체 생성 및 디자인 설정
dds <- DESeqDataSetFromMatrix(countData = counts_matrix,
                              colData = metadata,
                              design = ~ batch + target)

# 5. 정규화 (VST) 및 Batch Effect 제거
vsd <- vst(dds, blind = FALSE)
mat_vst <- assay(vsd)

# 분석 목적 변수인 target(암/정상)은 보존하고 batch만 제거
design <- model.matrix(~ target, data = metadata)
mat_corrected <- removeBatchEffect(mat_vst, batch = metadata$batch, design = design)

# 6. Python XGBoost가 먹기 좋은 형태로 변환 후 저장 [행=샘플, 열=유전자]
final_df <- as.data.frame(t(mat_corrected))
final_df$target <- ifelse(metadata$target == "cancer", 1, 0) # cancer=1, normal=0 변경
final_df$batch <- metadata$batch

# 전처리 완료된 파일 저장
write.csv(final_df, "corrected_data_for_xgboost.csv", row.names = TRUE)
print("R 단계 완료: TCGA ID 파싱 및 배치 제거 성공!")

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Installing package(s) 'limma'

also installing the dependency ‘statmod’



Attaching package: ‘limma’


The following object is masked from ‘package:DESeq2’:

    plotMA


The following object is masked from ‘package:BiocGenerics’:

    plotMA




[1] "--- 샘플 정보 자동 추출 완료 ---"

cancer normal 
    93     16 
[1] "R 단계 완료: TCGA ID 파싱 및 배치 제거 성공!"
